# LLM Classification Finetuning Baseline

This notebook explores the dataset, trains a baseline text classifier, evaluates holdout accuracy, and writes a submission CSV.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "competitions").exists() and (Path("/kaggle/working") / "competitions").exists():
    PROJECT_ROOT = Path("/kaggle/working")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

COMP_ROOT = PROJECT_ROOT / "competitions" / "llm_classification_finetuning"
RAW_DIR = COMP_ROOT / "data" / "raw"
RAW_DIR

In [ ]:
train_df = pd.read_csv(RAW_DIR / "train.csv")
test_df = pd.read_csv(RAW_DIR / "test.csv")
sample_df = pd.read_csv(RAW_DIR / "sample_submission.csv")

print(f"train shape: {train_df.shape}")
print(f"test shape: {test_df.shape}")
print(f"sample_submission shape: {sample_df.shape}")

display(train_df.head(3))
display(test_df.head(3))
display(sample_df.head(3))

In [ ]:
id_column = sample_df.columns[0]
target_column = [c for c in sample_df.columns if c != id_column][0]
print("id_column:", id_column)
print("target_column:", target_column)
if target_column in train_df.columns:
    display(train_df[target_column].value_counts(dropna=False).to_frame("count"))

In [ ]:
from competitions.llm_classification_finetuning.models.baseline import (
    build_dataset,
    discover_competition_files,
    fit_and_score_holdout,
    fit_final_model,
    generate_submission,
)

files = discover_competition_files(RAW_DIR)
dataset = build_dataset(files)

selection, holdout_metrics, holdout_predictions = fit_and_score_holdout(
    dataset=dataset,
    holdout_fraction=0.2,
    seed=42,
)

print("Selected strategy:", selection["selected_strategy"])
print("Holdout accuracy:", round(holdout_metrics["accuracy"], 6))
display(pd.DataFrame(holdout_metrics["strategy_metrics"]).sort_values("accuracy", ascending=False))
display(holdout_predictions.head(10))

In [ ]:
final_model = fit_final_model(dataset=dataset, selection=selection, seed=42)
submission = generate_submission(final_model, dataset=dataset)

submission_path = COMP_ROOT / "submissions" / "submission.csv"
submission_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(submission_path, index=False)

print("Saved submission:", submission_path)
display(submission.head(10))

Submit with Kaggle CLI:

```bash
kaggle competitions submit \
  -c llm-classification-finetuning \
  -f competitions/llm_classification_finetuning/submissions/submission.csv \
  -m "Baseline notebook run"
```